
# SDXL Ads Generator — Mac/MPS (Public Ads LoRA + Auto-switch to Your LoRA)

This notebook gives you **instant ad-style generations** using a **public SDXL Ads LoRA**, and will **auto-load your own LoRA** from `lora_adimage_sdxl/` when you train it.

**What you get:**
- Mac/MPS‑friendly inference (dtype handled after load, MPS safe)
- Loads **public Ads LoRA** `martintomov/sdxl-advertisement-lora` by default
- If your fine-tuned LoRA exists under `lora_adimage_sdxl/**.safetensors`, it will be used instead
- Utilities to generate **multiple banner sizes** with size-aware prompts


## 0) Install (run once, preferably in Terminal)

In [1]:

# Recommended installs (Mac wheels include MPS support in PyTorch)
# !python -m pip install --upgrade pip
# !pip install torch torchvision torchaudio
# !pip install diffusers transformers peft safetensors pillow

# If you see 'PEFT backend is required' during LoRA loading:
# !pip install -U peft


## 1) Config

In [2]:

MODEL_NAME = "stabilityai/stable-diffusion-xl-base-1.0"
PUBLIC_ADS_LORA = "martintomov/sdxl-advertisement-lora"  # pre-trained Ads LoRA on Hugging Face
OUTPUT_DIR = "lora_adimage_sdxl"                         # where your LoRA training will save .safetensors
RESOLUTION = 768                                         # generation size (use 512 if VRAM is tight)
OUTDIR = "samples"                                       # where images will be saved


## 2) Helpers (device, dtype, loading, generation)

In [3]:

import os, torch, math
from glob import glob
from typing import Tuple, List
from PIL import Image
from diffusers import AutoPipelineForText2Image

def get_device_dtype():
    use_mps = torch.backends.mps.is_available()
    device = "mps" if use_mps else "cpu"
    # MPS generally prefers fp16; switch to fp32 if you see numerical warnings
    dtype = torch.float16 if use_mps else torch.float32
    return device, dtype

def round_to_8(x: int) -> int:
    return int(round(x / 8) * 8)

def safe_cast_modules(pipe, dtype):
    # SDXL ignores dtype= at load; cast after loading
    try:
        pipe.unet.to(dtype=dtype)
        pipe.vae.to(dtype=dtype)
        if hasattr(pipe, "text_encoder"):    pipe.text_encoder.to(dtype=dtype)
        if hasattr(pipe, "text_encoder_2"):  pipe.text_encoder_2.to(dtype=dtype)
    except Exception as e:
        print("Note: module cast warning (safe to ignore):", e)

def load_pipeline(model_name: str):
    device, dtype = get_device_dtype()
    pipe = AutoPipelineForText2Image.from_pretrained(model_name).to(device)
    safe_cast_modules(pipe, dtype)
    # small memory helpers (xFormers not on MPS)
    try:
        pipe.enable_vae_tiling()
        pipe.enable_attention_slicing("max")
    except Exception:
        pass
    return pipe

def try_load_local_lora(pipe, output_dir: str) -> bool:
    """Return True if a local LoRA was found and loaded; otherwise False."""
    loras = sorted(glob(os.path.join(output_dir, "**", "*.safetensors"), recursive=True))
    if not loras:
        print(f"No local .safetensors in {output_dir}.")
        return False
    lora_path = loras[-1]
    lora_dir = os.path.dirname(lora_path)
    lora_file = os.path.basename(lora_path)
    print("Using LOCAL LoRA:", lora_path)
    pipe.load_lora_weights(lora_dir, weight_name=lora_file, adapter_name="ad_lora")
    pipe.set_adapters("ad_lora")
    return True

def load_ads_lora(pipe, repo_id: str):
    print("Loading PUBLIC Ads LoRA:", repo_id)
    pipe.load_lora_weights(repo_id, adapter_name="ad_lora_public")
    pipe.set_adapters("ad_lora_public")

def generate_banner(pipe, prompt_core: str, w: int, h: int, steps: int = 28, guidance: float = 5.0):
    # size-aware prompt composition
    prompt = f"banner {w}x{h}. {prompt_core}"
    negative = "blurry, low quality, jpeg artifacts, watermark, text cut off"
    # round dims to 8 (SDXL requirement)
    W, H = round_to_8(w), round_to_8(h)
    image = pipe(
        prompt,
        negative_prompt=negative,
        num_inference_steps=steps,
        guidance_scale=guidance,
        height=H, width=W,
    ).images[0]
    # exact resize ensures pixel-perfect unit
    return image.resize((w, h), Image.LANCZOS)

def generate_multi(pipe, prompt_core: str, sizes: List[Tuple[int,int]], outdir: str = "samples", tag: str = "ads"):
    os.makedirs(outdir, exist_ok=True)
    paths = []
    for (w, h) in sizes:
        img = generate_banner(pipe, prompt_core, w, h)
        fname = os.path.join(outdir, f"ad_{w}x{h}_{tag}.png")
        img.save(fname)
        print("Saved:", fname)
        paths.append(fname)
    return paths


## 3) Load SDXL + LoRA (uses your local LoRA if present, else public Ads LoRA)

In [4]:

pipe = load_pipeline(MODEL_NAME)

# Try local LoRA first (your fine-tuned weights). If not present, load public Ads LoRA.
if not try_load_local_lora(pipe, OUTPUT_DIR):
    load_ads_lora(pipe, PUBLIC_ADS_LORA)
    active_tag = "adsLORA"
else:
    active_tag = "yourLORA"

print("Active adapter tag:", active_tag)


Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

No local .safetensors in lora_adimage_sdxl.
Loading PUBLIC Ads LoRA: martintomov/sdxl-advertisement-lora


pytorch_lora_weights.safetensors:   0%|          | 0.00/23.4M [00:00<?, ?B/s]

No LoRA keys associated to CLIPTextModel found with the prefix='text_encoder'. This is safe to ignore if LoRA state dict didn't originally have any CLIPTextModel related params. You can also try specifying `prefix=None` to resolve the warning. Otherwise, open an issue if you think it's unexpected: https://github.com/huggingface/diffusers/issues/new
No LoRA keys associated to CLIPTextModelWithProjection found with the prefix='text_encoder_2'. This is safe to ignore if LoRA state dict didn't originally have any CLIPTextModelWithProjection related params. You can also try specifying `prefix=None` to resolve the warning. Otherwise, open an issue if you think it's unexpected: https://github.com/huggingface/diffusers/issues/new


Active adapter tag: adsLORA


## 4) Generate banners (multi-size example)

In [5]:

PROMPT_CORE = "clean promotional layout, bold headline, CTA button bottom-right, high contrast. summer shoe sale 20% off, free shipping."

# Choose sizes to generate
SIZES = [(300,250), (728,90), (160,600)]

paths = generate_multi(pipe, PROMPT_CORE, SIZES, outdir=OUTDIR, tag=active_tag)
paths[-1]


  0%|          | 0/28 [00:00<?, ?it/s]

Saved: samples/ad_300x250_adsLORA.png


  0%|          | 0/28 [00:00<?, ?it/s]

Saved: samples/ad_728x90_adsLORA.png


  0%|          | 0/28 [00:00<?, ?it/s]

Saved: samples/ad_160x600_adsLORA.png


'samples/ad_160x600_adsLORA.png'